# One-Click Batch Image Processor
Runs background removal, optional AI upscaling setup, resize to 1500x1000, and ZIP export.

In [ ]:
!pip -q install rembg onnxruntime pillow realesrgan
from google.colab import files
uploaded=files.upload()

In [ ]:
import os, zipfile, shutil
from PIL import Image
from rembg import remove

zip_name=list(uploaded.keys())[0]
shutil.rmtree("input",ignore_errors=True)
shutil.rmtree("output",ignore_errors=True)
os.makedirs("input"); os.makedirs("output")
with zipfile.ZipFile(zip_name) as z: z.extractall("input")

for fn in os.listdir("input"):
    p=os.path.join("input",fn)
    try:
        img=Image.open(p).convert("RGBA")
    except: continue
    fg=remove(img)
    # Placeholder upscale: enlarge before fitting. Replace with Real-ESRGAN call if desired.
    fg=fg.resize((fg.width*2, fg.height*2), Image.LANCZOS)
    canvas=Image.new("RGBA",(1500,1000),(0,0,0,0))
    ratio=min(1500/fg.width,1000/fg.height)
    fg=fg.resize((int(fg.width*ratio),int(fg.height*ratio)),Image.LANCZOS)
    x=(1500-fg.width)//2; y=(1000-fg.height)//2
    canvas.paste(fg,(x,y),fg)
    canvas.save(os.path.join("output",os.path.splitext(fn)[0]+".png"))
shutil.make_archive("processed_images","zip","output")
files.download("processed_images.zip")
print("Finished.")